# Building Segmentation usando Patch-Based Classification

## Progetto: Semantic Segmentation con CNN

**Dataset**: Massachusetts Buildings Dataset  
**Approccio**: Segmentation-by-Classification  
**Task**: Classificare ogni pixel come `building` (1) o `background` (0)

---

## Pipeline:
1. Caricamento dataset e visualizzazione
2. Estrazione patch bilanciate (3 TRUE + 3 FALSE per immagine)
3. Training CNN per classificazione patch
4. Dense prediction su immagine test
5. Valutazione e visualizzazioni

---
## 1. Setup e Import

In [ ]:
# Import librerie
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import skimage.io
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

# Deep Learning
import keras
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Activation

# Riproducibilità
np.random.seed(42)
random.seed(42)

print("✅ Import completati")

✅ Import completati


---
## 2. Configurazione e Caricamento Dataset

In [4]:
# Configurazione
PATCH_SIZE = 15  # Dimensione patch (15x15)
PATCHES_PER_CLASS = 3  # Numero di patch per classe per immagine
DATA_DIR = Path("archive")  # Modifica se necessario

# Paths
train_images_dir = DATA_DIR / "png" / "train"
train_labels_dir = DATA_DIR / "png" / "train_labels"
test_images_dir = DATA_DIR / "png" / "test"
test_labels_dir = DATA_DIR / "png" / "test_labels"

print(f"📁 Directory dataset: {DATA_DIR.absolute()}")
print(f"📊 Patch size: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"🎯 Patches per class: {PATCHES_PER_CLASS}")

📁 Directory dataset: /Users/davidecorso/Desktop/SUPSI/DeepLearning/Progetto/archive
📊 Patch size: 15x15
🎯 Patches per class: 3


In [5]:
def load_dataset(images_dir, labels_dir):
    """Carica tutte le immagini e le relative maschere da una directory"""
    images = []
    masks = []
    filenames = []
    
    image_paths = sorted(list(images_dir.glob("*.png")))
    
    print(f"Caricamento {len(image_paths)} immagini...")
    for img_path in tqdm(image_paths):
        # Carica immagine
        img = skimage.io.imread(img_path)
        
        # Carica maschera corrispondente
        mask_path = labels_dir / img_path.name
        mask = skimage.io.imread(mask_path)
        
        # Converti maschera in binaria (0 o 1)
        mask = (mask > 127).astype(np.uint8)  # Threshold a metà range
        
        images.append(img)
        masks.append(mask)
        filenames.append(img_path.name)
    
    return images, masks, filenames

# Carica training set
train_images, train_masks, train_filenames = load_dataset(train_images_dir, train_labels_dir)

# Carica test set
test_images, test_masks, test_filenames = load_dataset(test_images_dir, test_labels_dir)

print(f"\n✅ Training set: {len(train_images)} immagini")
print(f"✅ Test set: {len(test_images)} immagini")
print(f"📐 Dimensione immagini: {train_images[0].shape}")

Caricamento 137 immagini...


NameError: name 'tqdm' is not defined

---
## 3. Visualizzazione Dataset

In [ ]:
def visualize_samples(images, masks, filenames, num_samples=5, title="Samples"):
    """Visualizza immagini e maschere side-by-side"""
    indices = random.sample(range(len(images)), min(num_samples, len(images)))
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i, idx in enumerate(indices):
        # Immagine originale
        axes[i, 0].imshow(images[idx])
        axes[i, 0].set_title(f"Image: {filenames[idx]}")
        axes[i, 0].axis('off')
        
        # Maschera
        axes[i, 1].imshow(masks[idx], cmap='gray', vmin=0, vmax=1)
        axes[i, 1].set_title(f"Mask (Buildings)")
        axes[i, 1].axis('off')
        
        # Statistiche
        building_pixels = np.sum(masks[idx] == 1)
        total_pixels = masks[idx].size
        building_ratio = building_pixels / total_pixels * 100
        axes[i, 1].text(0.5, -0.1, f"Buildings: {building_ratio:.1f}%", 
                       transform=axes[i, 1].transAxes, ha='center')
    
    plt.suptitle(title, fontsize=16, y=1.001)
    plt.tight_layout()
    plt.show()

# Visualizza training samples
visualize_samples(train_images, train_masks, train_filenames, 
                  num_samples=5, title="Training Set Samples")

---
## 4. Estrazione Patch

**Strategia**:
- Per ogni immagine di training:
  - 3 patch centrate su pixel TRUE (building)
  - 3 patch centrate su pixel FALSE (background)
- Dataset finale bilanciato 50/50

In [ ]:
def extract_patch(image, center_y, center_x, patch_size):
    """
    Estrae una patch quadrata centrata su (center_y, center_x)
    
    Args:
        image: immagine sorgente
        center_y, center_x: coordinate del centro della patch
        patch_size: dimensione della patch (es. 15)
    
    Returns:
        patch estratta di dimensione (patch_size, patch_size, channels)
    """
    half_size = patch_size // 2
    
    # Calcola bounds
    y_start = center_y - half_size
    y_end = center_y + half_size + 1
    x_start = center_x - half_size
    x_end = center_x + half_size + 1
    
    # Estrai patch
    patch = image[y_start:y_end, x_start:x_end]
    
    return patch


def get_valid_pixel_coords(mask, class_value, patch_size):
    """
    Trova coordinate di pixel validi per estrarre patch
    (evita i bordi dove la patch non può essere estratta completamente)
    
    Args:
        mask: maschera binaria
        class_value: valore della classe da cercare (0 o 1)
        patch_size: dimensione della patch
    
    Returns:
        lista di coordinate (y, x) valide
    """
    half_size = patch_size // 2
    height, width = mask.shape
    
    # Trova tutti i pixel della classe richiesta
    coords = np.argwhere(mask == class_value)
    
    # Filtra solo quelli che permettono di estrarre una patch completa
    valid_coords = []
    for y, x in coords:
        if (half_size <= y < height - half_size and 
            half_size <= x < width - half_size):
            valid_coords.append((y, x))
    
    return valid_coords


def extract_patches_from_image(image, mask, patch_size, patches_per_class):
    """
    Estrae patch bilanciate da una singola immagine
    
    Args:
        image: immagine RGB
        mask: maschera binaria
        patch_size: dimensione delle patch
        patches_per_class: numero di patch da estrarre per classe
    
    Returns:
        patches: array di patch
        labels: array di label corrispondenti
    """
    patches = []
    labels = []
    
    # Estrai patch per classe 1 (building)
    building_coords = get_valid_pixel_coords(mask, class_value=1, patch_size=patch_size)
    if len(building_coords) >= patches_per_class:
        selected_coords = random.sample(building_coords, patches_per_class)
        for y, x in selected_coords:
            patch = extract_patch(image, y, x, patch_size)
            patches.append(patch)
            labels.append(1)
    
    # Estrai patch per classe 0 (background)
    background_coords = get_valid_pixel_coords(mask, class_value=0, patch_size=patch_size)
    if len(background_coords) >= patches_per_class:
        selected_coords = random.sample(background_coords, patches_per_class)
        for y, x in selected_coords:
            patch = extract_patch(image, y, x, patch_size)
            patches.append(patch)
            labels.append(0)
    
    return np.array(patches), np.array(labels)


def extract_patches_from_dataset(images, masks, patch_size, patches_per_class):
    """
    Estrae patch da tutto il dataset
    
    Returns:
        X: array di patch (N, patch_size, patch_size, 3)
        y: array di labels (N,)
    """
    all_patches = []
    all_labels = []
    
    print(f"Estrazione patch da {len(images)} immagini...")
    for img, mask in tqdm(zip(images, masks)):
        patches, labels = extract_patches_from_image(img, mask, patch_size, patches_per_class)
        all_patches.append(patches)
        all_labels.append(labels)
    
    X = np.vstack(all_patches)
    y = np.hstack(all_labels)
    
    return X, y


# Estrai patch da training set
print("\n=== ESTRAZIONE PATCH TRAINING SET ===")
X_train, y_train = extract_patches_from_dataset(
    train_images, train_masks, PATCH_SIZE, PATCHES_PER_CLASS
)

# Estrai patch da test set
print("\n=== ESTRAZIONE PATCH TEST SET ===")
X_test, y_test = extract_patches_from_dataset(
    test_images, test_masks, PATCH_SIZE, PATCHES_PER_CLASS
)

print(f"\n✅ Training patches: {X_train.shape}")
print(f"   - Classe 0 (background): {np.sum(y_train == 0)} patches")
print(f"   - Classe 1 (building): {np.sum(y_train == 1)} patches")
print(f"\n✅ Test patches: {X_test.shape}")
print(f"   - Classe 0 (background): {np.sum(y_test == 0)} patches")
print(f"   - Classe 1 (building): {np.sum(y_test == 1)} patches")

### Visualizzazione Patch Estratte

In [ ]:
def visualize_patches(X, y, num_samples=10):
    """Visualizza patches estratte"""
    # Prendi sample bilanciati
    class_0_indices = np.where(y == 0)[0]
    class_1_indices = np.where(y == 1)[0]
    
    n_per_class = num_samples // 2
    selected_0 = np.random.choice(class_0_indices, n_per_class, replace=False)
    selected_1 = np.random.choice(class_1_indices, n_per_class, replace=False)
    
    indices = np.concatenate([selected_0, selected_1])
    
    fig, axes = plt.subplots(2, num_samples // 2, figsize=(15, 6))
    
    for i, idx in enumerate(indices):
        row = i // (num_samples // 2)
        col = i % (num_samples // 2)
        
        axes[row, col].imshow(X[idx])
        label_name = "Building" if y[idx] == 1 else "Background"
        axes[row, col].set_title(f"{label_name}")
        axes[row, col].axis('off')
    
    plt.suptitle(f"Sample Patches ({PATCH_SIZE}x{PATCH_SIZE})", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_patches(X_train, y_train, num_samples=10)

---
## 5. Preprocessing e Normalizzazione

In [ ]:
# Normalizza patches in [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Converti labels in categorical (one-hot encoding)
y_train_cat = keras.utils.to_categorical(y_train, num_classes=2)
y_test_cat = keras.utils.to_categorical(y_test, num_classes=2)

print(f"✅ X_train normalizzato: {X_train_norm.shape}, range [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")
print(f"✅ y_train one-hot: {y_train_cat.shape}")
print(f"\nEsempio one-hot encoding:")
print(f"  Label originale: {y_train[:5]}")
print(f"  One-hot encoded:\n{y_train_cat[:5]}")

---
## 6. Costruzione e Training del Modello CNN

Architettura simile a quella vista con CIFAR-10, adattata per patch più piccole (15x15 invece di 32x32)

In [ ]:
def build_cnn_model(input_shape, num_classes=2):
    """
    Costruisce una CNN per classificazione di patch
    Simile all'architettura vista per CIFAR-10, ma più semplice
    """
    model = Sequential()
    
    # Primo blocco convoluzionale
    model.add(Conv2D(32, (3, 3), padding='same', input_shape=input_shape))
    model.add(Activation('relu'))
    model.add(Conv2D(32, (3, 3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    
    # Secondo blocco convoluzionale
    model.add(Conv2D(64, (3, 3), padding='same'))
    model.add(Activation('relu'))
    model.add(Conv2D(64, (3, 3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    
    # Fully connected layers
    model.add(Flatten())
    model.add(Dense(128))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes))
    model.add(Activation('softmax'))
    
    return model


# Costruisci il modello
model = build_cnn_model(input_shape=X_train_norm[0].shape, num_classes=2)

# Compila il modello
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Mostra l'architettura
model.summary()

In [ ]:
# Training del modello
EPOCHS = 20
BATCH_SIZE = 32

print(f"🚀 Inizio training per {EPOCHS} epoche...\n")

history = model.fit(
    X_train_norm, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test_norm, y_test_cat),
    verbose=1
)

print("\n✅ Training completato!")

### Visualizzazione Training History

In [ ]:
def plot_training_history(history):
    """Visualizza accuracy e loss durante il training"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Model Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

---
## 7. Evaluation: Metriche di Classificazione

Calcoliamo accuracy e AUC sul test set

In [ ]:
# Predizioni sul test set
y_pred_proba = model.predict(X_test_norm)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calcola metriche
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba[:, 1])
cm = confusion_matrix(y_test, y_pred)

print("="*50)
print("METRICHE DI CLASSIFICAZIONE")
print("="*50)
print(f"\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"📊 AUC-ROC: {auc:.4f}")
print(f"\n📊 Confusion Matrix:")
print(cm)
print(f"\n   True Negatives (BG → BG): {cm[0,0]}")
print(f"   False Positives (BG → Building): {cm[0,1]}")
print(f"   False Negatives (Building → BG): {cm[1,0]}")
print(f"   True Positives (Building → Building): {cm[1,1]}")

In [ ]:
# Visualizza confusion matrix
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Background', 'Building'],
            yticklabels=['Background', 'Building'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix\nAccuracy: {accuracy:.4f} | AUC: {auc:.4f}')
plt.show()

---
## 8. Dense Prediction: Segmentazione di Immagine Intera

Prendiamo un'immagine dal test set **non usata** per l'estrazione delle patch e applichiamo il modello densamente su ogni pixel.

In [ ]:
def dense_prediction(model, image, patch_size):
    """
    Applica il modello densamente su un'immagine intera
    
    Per ogni pixel (evitando i bordi), estrae una patch centrata
    e predice se il pixel centrale è building o background
    
    Args:
        model: modello CNN trainato
        image: immagine RGB di input
        patch_size: dimensione della patch
    
    Returns:
        predicted_mask: maschera booleana di segmentazione
        confidence_map: mappa di confidenza (probabilità classe 1)
    """
    height, width = image.shape[:2]
    half_size = patch_size // 2
    
    # Inizializza output
    predicted_mask = np.zeros((height, width), dtype=np.uint8)
    confidence_map = np.zeros((height, width), dtype=np.float32)
    
    # Lista per batch processing (più efficiente)
    patches = []
    positions = []
    
    print(f"Estrazione patches per predizione densa...")
    # Estrai tutte le patch
    for y in tqdm(range(half_size, height - half_size)):
        for x in range(half_size, width - half_size):
            patch = extract_patch(image, y, x, patch_size)
            patches.append(patch)
            positions.append((y, x))
    
    # Converti in numpy array e normalizza
    patches = np.array(patches).astype('float32') / 255.0
    
    print(f"Predizione su {len(patches)} patches...")
    # Predici in batch
    predictions = model.predict(patches, batch_size=256, verbose=1)
    
    print("Costruzione maschera finale...")
    # Riempi la maschera
    for (y, x), pred in zip(positions, predictions):
        class_pred = np.argmax(pred)
        confidence = pred[1]  # Probabilità classe building
        
        predicted_mask[y, x] = class_pred
        confidence_map[y, x] = confidence
    
    return predicted_mask, confidence_map


# Seleziona un'immagine dal test set
# IMPORTANTE: usa un'immagine diversa da quelle usate per estrarre le patch
# In questo caso, usiamo l'ultima immagine del test set
test_img_idx = -1  # Ultima immagine
test_image = test_images[test_img_idx]
test_mask_gt = test_masks[test_img_idx]
test_filename = test_filenames[test_img_idx]

print(f"\n🖼️  Immagine selezionata: {test_filename}")
print(f"📐 Dimensioni: {test_image.shape}\n")

# Applica dense prediction
predicted_mask, confidence_map = dense_prediction(model, test_image, PATCH_SIZE)

print("\n✅ Dense prediction completata!")

### Visualizzazione Risultati

In [ ]:
def visualize_segmentation_results(image, ground_truth, prediction, confidence, filename):
    """
    Visualizza i risultati della segmentazione
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Immagine originale
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Ground truth
    axes[0, 1].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
    axes[0, 1].set_title('Ground Truth Mask', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Predizione
    axes[0, 2].imshow(prediction, cmap='gray', vmin=0, vmax=1)
    axes[0, 2].set_title('Predicted Mask', fontsize=14, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Overlay ground truth
    overlay_gt = image.copy()
    overlay_gt[ground_truth == 1] = [255, 0, 0]  # Rosso per buildings
    axes[1, 0].imshow(overlay_gt)
    axes[1, 0].set_title('GT Overlay (Red = Buildings)', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Overlay predizione
    overlay_pred = image.copy()
    overlay_pred[prediction == 1] = [0, 255, 0]  # Verde per buildings predetti
    axes[1, 1].imshow(overlay_pred)
    axes[1, 1].set_title('Prediction Overlay (Green = Buildings)', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Confidence map
    im = axes[1, 2].imshow(confidence, cmap='hot', vmin=0, vmax=1)
    axes[1, 2].set_title('Confidence Map (Building Probability)', fontsize=14, fontweight='bold')
    axes[1, 2].axis('off')
    plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)
    
    plt.suptitle(f'Segmentation Results: {filename}', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()


# Visualizza i risultati
visualize_segmentation_results(
    test_image, test_mask_gt, predicted_mask, confidence_map, test_filename
)

### Calcolo Metriche di Segmentazione

In [ ]:
def calculate_iou(ground_truth, prediction):
    """Calcola Intersection over Union (IoU) / Jaccard Index"""
    intersection = np.logical_and(ground_truth, prediction).sum()
    union = np.logical_or(ground_truth, prediction).sum()
    
    if union == 0:
        return 1.0  # Se entrambi sono vuoti, IoU perfetto
    
    return intersection / union


def calculate_dice(ground_truth, prediction):
    """Calcola Dice Coefficient / F1 Score"""
    intersection = np.logical_and(ground_truth, prediction).sum()
    
    if ground_truth.sum() + prediction.sum() == 0:
        return 1.0
    
    return 2 * intersection / (ground_truth.sum() + prediction.sum())


# Calcola metriche solo sulla regione valida (dove abbiamo fatto predizioni)
half_size = PATCH_SIZE // 2
gt_valid = test_mask_gt[half_size:-half_size, half_size:-half_size]
pred_valid = predicted_mask[half_size:-half_size, half_size:-half_size]

# Calcola metriche
pixel_accuracy = np.mean(gt_valid == pred_valid)
iou = calculate_iou(gt_valid, pred_valid)
dice = calculate_dice(gt_valid, pred_valid)

print("="*50)
print("METRICHE DI SEGMENTAZIONE")
print("="*50)
print(f"\n🎯 Pixel Accuracy: {pixel_accuracy:.4f} ({pixel_accuracy*100:.2f}%)")
print(f"🎯 IoU (Jaccard): {iou:.4f}")
print(f"🎯 Dice Coefficient: {dice:.4f}")
print(f"\n📊 Building pixels in GT: {gt_valid.sum()} ({gt_valid.sum()/gt_valid.size*100:.2f}%)")
print(f"📊 Building pixels predicted: {pred_valid.sum()} ({pred_valid.sum()/pred_valid.size*100:.2f}%)")

---
## 9. Applicazione su Più Immagini Test

In [ ]:
# Applica dense prediction su 3 immagini test diverse
num_test_images = min(3, len(test_images))

for i in range(num_test_images):
    print(f"\n{'='*60}")
    print(f"IMMAGINE TEST {i+1}/{num_test_images}: {test_filenames[i]}")
    print(f"{'='*60}\n")
    
    # Dense prediction
    pred_mask, conf_map = dense_prediction(model, test_images[i], PATCH_SIZE)
    
    # Visualizza
    visualize_segmentation_results(
        test_images[i], test_masks[i], pred_mask, conf_map, test_filenames[i]
    )
    
    # Metriche
    half_size = PATCH_SIZE // 2
    gt_valid = test_masks[i][half_size:-half_size, half_size:-half_size]
    pred_valid = pred_mask[half_size:-half_size, half_size:-half_size]
    
    pixel_acc = np.mean(gt_valid == pred_valid)
    iou_score = calculate_iou(gt_valid, pred_valid)
    dice_score = calculate_dice(gt_valid, pred_valid)
    
    print(f"Pixel Accuracy: {pixel_acc:.4f} | IoU: {iou_score:.4f} | Dice: {dice_score:.4f}")

---
## 10. Salvataggio Modello e Risultati

In [ ]:
# Salva il modello
model.save('building_segmentation_model.h5')
print("✅ Modello salvato: building_segmentation_model.h5")

# Salva anche l'ultima predizione
plt.imsave('predicted_mask.png', predicted_mask, cmap='gray')
plt.imsave('confidence_map.png', confidence_map, cmap='hot')
print("✅ Maschere salvate: predicted_mask.png, confidence_map.png")

---
## Conclusioni e Next Steps

### ✅ Cosa abbiamo fatto:
1. Caricato dataset Massachusetts Buildings con 2 classi (building/background)
2. Estratto patch bilanciate (3 per classe per immagine)
3. Trainato una CNN per classificare patch
4. Applicato il modello densamente su immagini intere
5. Valutato con metriche di classificazione (Accuracy, AUC) e segmentazione (IoU, Dice)

### 🚀 Possibili miglioramenti:
1. **Aumentare patch size** (es. 31x31, 51x51) per catturare più contesto
2. **Data augmentation** (rotation, flip, brightness)
3. **Più patch per immagine** durante training
4. **Architetture più complesse** (es. più layer, residual connections)
5. **Post-processing** (smoothing, CRF) per maschere più pulite
6. **Fully Convolutional Networks (FCN)** per segmentazione diretta

### 📊 Valutazione progetto:
- ✅ Visualizzazioni chiare e informative
- ✅ Pipeline ben strutturato
- ✅ Metriche appropriate calcolate
- ✅ Codice documentato e riproducibile